# Notebook 04 -- A. palmata connectivity analysis

Aim (see comment by IBB in Apal PopGen manuscript draft):
> estimate number of migrants per generation

**Source:** Nick's table of pairwise Fst (`table2.fst.xlsx`)

### Imports

In [ ]:
import sys
sys.executable
from pathlib import Path

import importlib.util
import subprocess

from datetime import datetime
import getpass
import socket
import hashlib

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from scipy import stats
from scipy.spatial.distance import squareform



In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 150)

In [ ]:
PROJECT_DIR = Path('/Users/kidelu001/awi-work/science/projects/apal_popgen/')

DATA_DIR = PROJECT_DIR / 'data'
DERIV_DIR = PROJECT_DIR / 'deriv' / 'manuscript'
META_DIR = PROJECT_DIR / 'metadata'

PLOTS_DIR = PROJECT_DIR / 'output' / 'plots' / 'n_migrants'


In [ ]:
APAL_IBD_TSV = DERIV_DIR / 'sample_registry' / 'apal_ibd.tsv'
META_TSV = PROJECT_DIR / 'output' / 'tables' / 'suptable1_merged.tsv'


---
### Initialize log

In [ ]:
NOTEBOOK_RUN_TIMESTAMP = datetime.now().astimezone()
NOTEBOOK_NAME = "04_apal_fst_migrants.ipynb"

print("Notebook:", NOTEBOOK_NAME)
print("User:", getpass.getuser())
print("Host:", socket.gethostname())
print("Timestamp:", NOTEBOOK_RUN_TIMESTAMP.isoformat())


---
### Variables

---
### Load

In [ ]:
meta = pd.read_csv(META_TSV, sep="\t")
meta = meta.loc[meta["region_sub"] != "State College"].copy()

meta.head()

In [ ]:
meta.groupby(["region_sub"]).size()

In [ ]:
ID_COL = "affymetrix_id"
GENET_COL = "mlg"
MLG_COL = "mlg"
REGION_COL = "region"
SUBREGION_COL = "region_sub"
SITE_COL = "reef"
REEF_COL = "reef"
LAT_COL = "lat_clean"
LON_COL = "lon_clean"

In [ ]:
# 1) unique genets per region
genets_per_region = (
    meta.groupby("region_sub", dropna=False)[GENET_COL]
    .nunique()
    .sort_values(ascending=False)
    .rename("n_genets")
    .reset_index()
)
genets_per_region

In [ ]:
def classify_pair(row):
    pair = {row["region1"], row["region2"]}
    if pair <= {"Mexico", "Belize", "Honduras"}:
        return "MesoAm"
    if pair <= {"Mexico_Gulf"}:
        return "GoM"
    if pair <= {"Florida", "Bahamas"}:
        return "Northern"
    if pair <= {"Cuba", "Jamaica", "Dominican Republic", "Puerto Rico", "USVI"}:
        return "Greater Antilles"
    if pair <= {"Curacao", "Colombia"}:
        return "Southern"
    if pair <= {"Colombia_SanAndres"}:
        return "SA"
    return "Cross-basin"

In [ ]:
# get region centroids
region_coords = (
    meta.groupby("region_sub", dropna=False)
    .agg(
        lat=("lat_clean", "median"),
        lon=("lon_clean", "median"),
    )
    .reset_index()
    .rename(columns={"region_final": "region"})
)

region_coords

In [ ]:
from math import radians, sin, cos, sqrt, atan2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return r * c

In [ ]:
import matplotlib as mpl

In [ ]:
from matplotlib.lines import Line2D

---
# Re-do with Nick's table of pairwise Fst

In [ ]:
xlsx_path = PROJECT_DIR / "output" / "tables"/ "table2.fst.xlsx"

In [ ]:
nick_raw = pd.read_excel(xlsx_path, index_col=0)
nick_raw

In [ ]:
regions = nick_raw.index.tolist()

fst_mat_nick = pd.DataFrame(np.nan, index=regions, columns=regions, dtype=float)
ci_mat_nick = pd.DataFrame("", index=regions, columns=regions, dtype="string")

for i, r1 in enumerate(regions):
    for j, r2 in enumerate(regions):
        val = nick_raw.loc[r1, r2]

        if i == j:
            fst_mat_nick.loc[r1, r2] = 0.0
            ci_mat_nick.loc[r1, r2] = ""
        elif i > j:
            # lower triangle: FST values
            fst_mat_nick.loc[r1, r2] = pd.to_numeric(val, errors="coerce")
            fst_mat_nick.loc[r2, r1] = pd.to_numeric(val, errors="coerce")
        elif i < j:
            # upper triangle: CI strings
            ci_mat_nick.loc[r1, r2] = "" if pd.isna(val) else str(val).replace("\n", " ")

In [ ]:
regions

In [ ]:
region_rename = {
    "BH": "Bahamas",
    "BZ": "Belize",
    "CO": "Colombia",
    "SA": "Col_SA",
    "CB": "Cuba",
    "CU": "Curacao",
    "DR": "DR",
    "FL": "Florida",
    "HN": "Honduras",
    "JM": "Jamaica",
    "GOM": "Mex_Gulf",
    "MX": "Mexico",
    "PR": "PR",
    "VI": "USVI",
}

fst_mat_nick = fst_mat_nick.rename(index=region_rename, columns=region_rename)
ci_mat_nick = ci_mat_nick.rename(index=region_rename, columns=region_rename)

In [ ]:
n_mat_nick = pd.DataFrame(np.nan, index=fst_mat_nick.index, columns=fst_mat_nick.columns, dtype=float)

for r1 in fst_mat_nick.index:
    for r2 in fst_mat_nick.columns:
        fst = fst_mat_nick.loc[r1, r2]

        if r1 == r2:
            n_mat_nick.loc[r1, r2] = np.nan
        elif pd.notna(fst) and (fst > 0) and (fst < 1):
            n_mat_nick.loc[r1, r2] = (1 - fst) / (4 * fst)

n_mat_nick

In [ ]:
cluster_source = fst_mat_nick.copy()
np.fill_diagonal(cluster_source.values, 0.0)

condensed = squareform(cluster_source.to_numpy(), checks=False)
Z = linkage(condensed, method="average")
order = leaves_list(Z)

ordered_regions = fst_mat_nick.index[order].tolist()
ordered_regions

#### Re-order matrices

In [ ]:
# Build geographic distance matrix for Nick's regions
geo_mat_nick = pd.DataFrame(np.nan, index=fst_mat_nick.index, columns=fst_mat_nick.columns, dtype=float)
region_lookup_nick = region_coords.set_index("region_sub")

for r1 in fst_mat_nick.index:
    for r2 in fst_mat_nick.columns:
        if r1 == r2:
            geo_mat_nick.loc[r1, r2] = np.nan
        else:
            geo_mat_nick.loc[r1, r2] = haversine_km(
                region_lookup_nick.loc[r1, "lat"],
                region_lookup_nick.loc[r1, "lon"],
                region_lookup_nick.loc[r2, "lat"],
                region_lookup_nick.loc[r2, "lon"],
            )

fst_ord_nick = fst_mat_nick.loc[ordered_regions, ordered_regions]
n_ord_nick   = n_mat_nick.loc[ordered_regions, ordered_regions]
geo_ord_nick = geo_mat_nick.loc[ordered_regions, ordered_regions]

In [ ]:
n_ord_round = n_ord_nick.round(0)
geo_ord_round = geo_ord_nick.round(0)

mirrored_text_nick = pd.DataFrame(
    "",
    index=n_ord_round.index,
    columns=n_ord_round.columns,
    dtype="string",
)

for i, r1 in enumerate(mirrored_text_nick.index):
    for j, r2 in enumerate(mirrored_text_nick.columns):
        if i < j:
            val = n_ord_round.loc[r1, r2]
            mirrored_text_nick.loc[r1, r2] = "" if pd.isna(val) else f"{int(val)}"
        elif i > j:
            val = geo_ord_round.loc[r1, r2]
            mirrored_text_nick.loc[r1, r2] = "" if pd.isna(val) else f"{int(val)}"
        else:
            mirrored_text_nick.loc[r1, r2] = "—"

# mirrored_text_nick.index.name = "region"
mirrored_text_nick

## Plot

In [ ]:
region_lookup = region_coords.set_index("region_sub")

In [ ]:
region_coords = region_coords.copy()

region_coords["region_sub"] = region_coords["region_sub"].replace({
    "Mexico_Gulf": "Mex_Gulf",
    "Colombia_SanAndres": "Col_SA",
    "Dominican Republic": "DR",
    "Puerto Rico": "PR",
})

In [ ]:
def fst_mat_to_nm_long(
    fst_mat,
    region_coords,
    region_col="region",
    lat_col="lat",
    lon_col="lon",
    pair_group_map=None,
):
    """
    Convert a symmetric FST matrix to a long table with:
    - fst
    - n_migrants
    - log10_n_migrants
    - geo_dist_km
    - optional pair_group
    """
    region_lookup = region_coords.set_index(region_col)
    rows = []

    for i, r1 in enumerate(fst_mat.index):
        for j, r2 in enumerate(fst_mat.columns):
            if i >= j:
                continue

            fst = fst_mat.loc[r1, r2]

            nm = np.nan
            log10_nm = np.nan
            if pd.notna(fst) and (fst > 0) and (fst < 1):
                nm = (1 - fst) / (4 * fst)
                if nm > 0:
                    log10_nm = np.log10(nm)

            geo_dist = haversine_km(
                region_lookup.loc[r1, lat_col],
                region_lookup.loc[r1, lon_col],
                region_lookup.loc[r2, lat_col],
                region_lookup.loc[r2, lon_col],
            )

            row = {
                "region1": r1,
                "region2": r2,
                "pair": f"{r1} – {r2}",
                "fst": fst,
                "n_migrants": nm,
                "log10_n_migrants": log10_nm,
                "geo_dist_km": geo_dist,
            }

            if pair_group_map is not None:
                g1 = pair_group_map.get(r1, "Other")
                g2 = pair_group_map.get(r2, "Other")
                row["group1"] = g1
                row["group2"] = g2
                row["pair_group"] = g1 if g1 == g2 else "Cross-basin"

            rows.append(row)

    return pd.DataFrame(rows)


def plot_nm_vs_distance(
    plot_df,
    y_col="log10_n_migrants",
    pair_group_colors=None,
    label_sd_mult=1.5,
    # label_top_n=5,
    label_side="both",
    label_offsets=None,
    default_label_style=None,
    point_size=80,
    point_edgecolor="black",
    point_linewidth=0.8,
    alpha=0.85,
    figsize=(7, 7),
    title=None,
    xlabel="Geographic distance (km)",
    ylabel=None,
    save_base=None,
):
    """
    Scatter plot of implied migrants vs distance with OLS fit and outlier labels.

    Outliers are defined as points whose residuals exceed label_sd_mult * SD(residuals).

    label_side:
      - "both"  -> label unusually high or low residuals
      - "above" -> label only points above the regression line
      - "below" -> label only points below the regression line

    label_offsets can be a dict keyed by row['pair'], with values like:
    {"Belize – Honduras": {"dx": 20, "dy": 0.03, "ha": "left", "va": "bottom"}}
    """
    if pair_group_colors is None:
        pair_group_colors = {"Cross-group": "#888888"}

    if label_offsets is None:
        label_offsets = {}

    if default_label_style is None:
        default_label_style = {"dx": 8, "dy": 0.0, "ha": "left", "va": "center"}

    df = plot_df.copy()
    valid = np.isfinite(df["geo_dist_km"]) & np.isfinite(df[y_col])
    df = df.loc[valid].copy()

    if df.empty:
        raise ValueError(f"No finite values found for y_col='{y_col}' and geo_dist_km.")

    ols = stats.linregress(df["geo_dist_km"], df[y_col])
    df["pred"] = ols.slope * df["geo_dist_km"] + ols.intercept
    df["resid"] = df[y_col] - df["pred"]
    df["abs_resid"] = df["resid"].abs()

    resid_sd = df["resid"].std()
    cutoff = label_sd_mult * resid_sd

    if label_side == "both":
        label_df = df.loc[df["abs_resid"] >= cutoff].copy()
    elif label_side == "above":
        label_df = df.loc[df["resid"] >= cutoff].copy()
    elif label_side == "below":
        label_df = df.loc[df["resid"] <= -cutoff].copy()
    else:
        raise ValueError("label_side must be one of: 'both', 'above', 'below'")

    fig, ax = plt.subplots(figsize=figsize)

    group_col = "pair_group" if "pair_group" in df.columns else None

    if group_col is not None:
        for group, subdf in df.groupby(group_col):
            ax.scatter(
                subdf["geo_dist_km"],
                subdf[y_col],
                label=group,
                alpha=alpha,
                s=point_size,
                c=pair_group_colors.get(group, "#888888"),
                edgecolors=point_edgecolor,
                linewidths=point_linewidth,
            )
    else:
        ax.scatter(
            df["geo_dist_km"],
            df[y_col],
            alpha=alpha,
            s=point_size,
            c="#888888",
            edgecolors=point_edgecolor,
            linewidths=point_linewidth,
        )

    xline = np.linspace(df["geo_dist_km"].min(), df["geo_dist_km"].max(), 200)
    ax.plot(
        xline,
        ols.slope * xline + ols.intercept,
        linewidth=1,
        color="grey",
    )
    print(label_df)
    
    for _, row in label_df.iterrows():
        style = default_label_style.copy()
        style.update(label_offsets.get(row["pair"], {}))

        ax.text(
            row["geo_dist_km"] + style["dx"],
            row[y_col] + style["dy"],
            row["pair"],
            fontsize=8,
            alpha=0.9,
            ha=style["ha"],
            va=style["va"],
        )

    if ylabel is None:
        ylabel = "log10 implied Nm" if y_col == "log10_n_migrants" else "Implied Nm"

    if title is None:
        title = "Region-pair connectivity vs distance"

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

    if group_col is not None:
        ax.legend(frameon=False)

    plt.tight_layout()

    if save_base is not None:
        save_base = str(save_base)
        fig.savefig(f"{save_base}.pdf", bbox_inches="tight")
        fig.savefig(f"{save_base}.ps", format="ps", bbox_inches="tight")
        fig.savefig(f"{save_base}.eps", format="eps", bbox_inches="tight")

    plt.show()

    return {
        "plot_df": df,
        "label_df": label_df,
        "ols": ols,
        "r2": ols.rvalue ** 2,
        "resid_sd": resid_sd,
        "cutoff": cutoff,
    }

In [ ]:
# define broad groups
region_group_map = {
    "Bahamas": "Northern",
    "Florida": "Northern",
    "Mexico": "MesoAm",
    "Belize": "MesoAm",
    "Mex_Gulf": "Mex_Gulf",
    "Honduras": "MesoAm",
    "Cuba": "Greater Antilles",
    "Jamaica": "Greater Antilles",
    "Dominican Republic": "Greater Antilles",
    "Puerto Rico": "Greater Antilles",
    "DR": "Greater Antilles",
    "PR": "Greater Antilles",
    "USVI": "Greater Antilles",
    "Curacao": "Southern",
    "Colombia": "Southern",
    "Col_SA": "Col_SA"
}

pair_group_colors = {
    "Northern": "darkorange", 
    "Greater Antilles": "cornflowerblue",
    "Southern": "yellowgreen",
    "MesoAm": "darkorchid",
    "Mex_Gulf": "indianred",
    "Col_SA": "forestgreen",
    "Cross-basin": "grey",
}

nick_pairs = fst_mat_to_nm_long(
    fst_mat=fst_mat_nick,        # your Nick-based symmetric FST matrix
    region_coords=region_coords, # must contain region, lat, lon
    region_col="region_sub",
    lat_col="lat",
    lon_col="lon",
    pair_group_map=region_group_map,
)

In [ ]:
dx = 21
dy = 0.02
label_offsets_log = {
    "Belize – Honduras": {"dx": dx, "dy": dy, "ha": "left", "va": "bottom"},
    "Honduras – Mexico": {"dx": dx, "dy": dy, "ha": "left", "va": "bottom"},
    "Belize – Mexico": {"dx": dx, "dy": -0.04, "ha": "left", "va": "top"},
    "Florida – Mexico": {"dx": dx, "dy": dy, "ha": "left", "va": "bottom"},
    "Belize – Florida": {"dx": dx, "dy": dy, "ha": "left", "va": "bottom"},
    "Florida – Honduras": {"dx": dx, "dy": -0.04, "ha": "left", "va": "top"},
    "Colombia – Curacao": {"dx": 20.5, "dy": -0.06, "ha": "left", "va": "top"},
}


res_log = plot_nm_vs_distance(
    nick_pairs,
    y_col="log10_n_migrants",
    pair_group_colors=pair_group_colors,
    label_sd_mult=1.5,
    label_side="both",
    label_offsets=label_offsets_log,
    title="Region-pair connectivity vs distance",
    ylabel="log10 implied Nm",
    save_base=PLOTS_DIR / "scatter_log10migrants_vs_distance_NL_outliers=SD1,5",
)

In [ ]:
dx = 50
dy = 2

label_offsets_raw = {
    "Belize – Honduras": {"dx": 50, "dy": 2, "ha": "left", "va": "bottom"},
    "Honduras – Mexico": {"dx": dx, "dy": dy, "ha": "left", "va": "bottom"},
    "Belize – Mexico": {"dx": dx, "dy": -dy, "ha": "left", "va": "top"},
    "Florida – Mexico": {"dx": dx, "dy": dy, "ha": "left", "va": "bottom"},
    "Belize – Florida": {"dx": dx, "dy": dy, "ha": "left", "va": "bottom"},
    "Florida – Honduras": {"dx": dx, "dy": -dy, "ha": "left", "va": "top"},
    "Colombia – Curacao": {"dx": dx, "dy": -dy, "ha": "left", "va": "top"},
}
res_raw = plot_nm_vs_distance(
    nick_pairs,
    y_col="n_migrants",
    pair_group_colors=pair_group_colors,
    label_sd_mult=2.0,
    label_side="both",
    label_offsets=label_offsets_raw,
    title="Region-pair connectivity vs distance",
    ylabel="Implied Nm",
    # save_base=PLOTS_DIR / "scatter_migrants_vs_distance_NL_outliers=SD2,0",
    save_base=None
)

In [ ]:
plot_df = nick_pairs.copy()

plot_df["pair_scope"] = np.where(
    plot_df["group1"] == plot_df["group2"],
    "within",
    "between",
)

plot_df = plot_df.sort_values("log10_n_migrants").reset_index(drop=True).copy()

scope_style = {
    "within": {
        "facecolors": "dimgrey",
        "edgecolors": "dimgrey",
    },
    "between": {
        "facecolors": "whitesmoke",
        "edgecolors": "dimgrey",
    },
}

fig, ax = plt.subplots(figsize=(8,12))

for _, row in plot_df.iterrows():
    style = scope_style.get(
        row["pair_scope"],
        {"facecolors": "grey", "edgecolors": "dimgrey"},
    )

    ax.scatter(
        row["log10_n_migrants"],
        row["pair"],
        facecolors=style["facecolors"],
        edgecolors=style["edgecolors"],
        linewidths=0.8,
        alpha=0.9,
    )

ax.set_xlabel("Implied Nm (log10)")
ax.set_ylabel("")
ax.set_title("Region-pair estimated number of migrants")
ax.grid(True, axis="x", alpha=0.3)
ax.tick_params(axis="y", labelsize=6)
ax.tick_params(axis="x", labelsize=8)

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markerfacecolor="dimgrey",
        markeredgecolor="dimgrey",
        label="within",
        markersize=7,
    ),
    Line2D(
        [0], [0],
        marker="o",
        linestyle="",
        markerfacecolor="whitesmoke",
        markeredgecolor="dimgrey",
        label="between",
        markersize=7,
    ),
]

ax.legend(handles=legend_handles, frameon=False)

plt.tight_layout()

out_base = PLOTS_DIR / "dot_cleveland_region_pair_migrants_log10_NL"

fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(out_base.with_suffix(".ps"), format="ps", bbox_inches="tight")
fig.savefig(out_base.with_suffix(".eps"), format="eps", bbox_inches="tight")

plt.show()

In [ ]:
# --- Clustermap: Fst (upper triangle) vs log10 Nm (lower triangle) ---
log10_n_ord_nick = np.log10(n_ord_nick.replace(0, np.nan))

n = len(fst_ord_nick)
mask_upper = np.tril(np.ones((n, n), dtype=bool), k=0)
mask_lower = np.triu(np.ones((n, n), dtype=bool), k=0)

fig = plt.figure(figsize=(10.5, 8.5))
ax      = fig.add_axes([0.24, 0.14, 0.56, 0.56])
cax_nm  = fig.add_axes([0.12, 0.28, 0.018, 0.28])  # left colorbar
cax_fst = fig.add_axes([0.84, 0.28, 0.018, 0.28])  # right colorbar

fst_vmin = np.nanmin(fst_ord_nick.to_numpy())
fst_vmax = np.nanmax(fst_ord_nick.to_numpy())
nm_vmin  = np.nanmin(log10_n_ord_nick.to_numpy())
nm_vmax  = np.nanmax(log10_n_ord_nick.to_numpy())

cmap_fst = mpl.colormaps["mako_r"]
cmap_nm  = mpl.colormaps["viridis"]
norm_fst = mpl.colors.Normalize(vmin=fst_vmin, vmax=fst_vmax)
norm_nm  = mpl.colors.Normalize(vmin=nm_vmin,  vmax=nm_vmax)

sns.heatmap(
    fst_ord_nick,
    mask=mask_upper,
    cmap=cmap_fst, norm=norm_fst,
    square=True, linewidths=0.5, linecolor="white",
    cbar=False, ax=ax,
)
sns.heatmap(
    log10_n_ord_nick,
    mask=mask_lower,
    cmap=cmap_nm, norm=norm_nm,
    square=True, linewidths=0.5, linecolor="white",
    cbar=False, ax=ax,
)

sm_nm  = mpl.cm.ScalarMappable(cmap=cmap_nm,  norm=norm_nm);  sm_nm.set_array([])
sm_fst = mpl.cm.ScalarMappable(cmap=cmap_fst, norm=norm_fst); sm_fst.set_array([])
cb_nm  = fig.colorbar(sm_nm,  cax=cax_nm,  orientation="vertical")
cb_fst = fig.colorbar(sm_fst, cax=cax_fst, orientation="vertical")
cb_nm.set_label("log10 implied Nm", rotation=90, labelpad=10)
cb_fst.set_label("FST",            rotation=90, labelpad=10)
cb_nm.ax.tick_params(labelsize=8)
cb_fst.ax.tick_params(labelsize=8)

ax.set_title("Pairwise FST (upper) and implied connectivity log10 Nm (lower) — Nick's Fst", pad=18)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=90)
ax.tick_params(axis="y", rotation=0)

out_base = PLOTS_DIR / "clustermap_fst_migrants_NL"
fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(out_base.with_suffix(".ps"),  format="ps",  bbox_inches="tight")
fig.savefig(out_base.with_suffix(".eps"), format="eps", bbox_inches="tight")
plt.show()
